# Post process and display results for LLM models


In [1]:
import json

In [5]:
#load results
MODEL_NAME =  "gpt-4o-mini"
savename = 'test_predict_llm'+MODEL_NAME+'.json'
file_path = '../Data_backup/results_llm/'+savename
# Open and read the JSON file
with open(file_path, 'r') as json_file:
    test_results_json = json.load(json_file)

In [159]:
test_results_json

{'MDRDZ011': {'input_text': ['DREF Operation Algeria Flood 2024 Bechar Appeal MDRDZ011 Country Algeria Hazard Flood Type of DREF Response Crisis Category Yellow Event Onset Sudden DREF Allocation CHF 499,186 Glide Number FL2024000168DZA People Affected 11,100 people People Targeted 6,000 people Operation Start Date 19092024 Operation Timeframe 6 months Operation End Date 31032025 DREF Published 22092024 Targeted Areas Bchar, Tamanrasset, El Bayadh Page 1 13 Description of the Event Date of event 08092024 What happened, where and when?',
   'On September 8, 2024, a severe tropical disturbance triggered widespread flooding across several provinces in southern and western Algeria.',
   'The most affected areas include Bchar, Elbayadh, Beni Abbes, Tamanrasset, Tiaret, Tindouf, and Naama.',
   'The flooding, which began around September 5th, intensified by September 8th, displacing approximately 2,220 families, some of them from nomadic communities.',
   'In Bchar, the number of displaced f

In [150]:
#post process the results
import re
import copy as cp
import datetime as datetime
def clean_output(results_json):
    dict_reports={}
    for report_id, report in results_json.items():
        results = report['results']
        results_process = cp.deepcopy(results)
        results_process = re.sub('[\{\}]', '', results_process)
        results_process= results_process.split("\n")
        results_process = results_process[1: -1]
        dict_results = dict()
        for pair in results_process:
            if len(pair.split(":")) == 2:
                key, value = pair.split(":")
            else:
                print(pair.split(":"))
                break
            key = re.sub("[\", ]","",key)
            value = re.sub("[\"]","",value)
            if key=='Location':
                value = list(value.split(','))
                value = [re.sub("[\,, ]","",ival) for ival in value]
            elif key=='Start_Date' or key=='End_Date':
                value = value.replace(",","")
            elif key=='Sentences':
                pass
            else:
                value = re.sub("[\,, ]","",value)
            dict_results[key] = value
        dict_reports[report_id] = dict_results
    return dict_reports

In [151]:
dict_processed = clean_output(test_results_json)

['            "Pakistan has experienced an unusually intense and prolonged monsoon season, resulting in widespread infrastructure damage, numerous casualties, and significant injuries.",']
['    "Cameroons Far North region has been experiencing flooding since the start of the rainy season, which began in the second half of July.",']
['    "Intense rainfall observed in the departments of Mono, Couffo, Zou and Oum in the South of Benin caused the overflow of the river Couffo on 26 June 2024 in 6 of the 11 districts of the commune it crosses in Couffo department.",']


In [152]:
dict_processed

{'MDRDZ011': {'Hazard': 'Flood',
  'Country': 'Algeria',
  'Location': ['Bchar',
   'Elbayadh',
   'BeniAbbes',
   'Tamanrasset',
   'Tiaret',
   'Tindouf',
   'Naama',
   ''],
  'Start_Date': ' September 5 2024',
  'End_Date': ' NULL',
  'Sentences': ' On September 8, 2024, a severe tropical disturbance triggered widespread flooding across several provinces in southern and western Algeria. The flooding, which began around September 5th, intensified by September 8th, displacing approximately 2,220 families, some of them from nomadic communities.'},
 'MDRPK026': {'Hazard': 'Flood',
  'Country': 'Pakistan',
  'Location': ['Jacobabad',
   'NaushahroFeroz',
   'Ghotki',
   'Sukkur',
   'Sanghar',
   'Dadu',
   'ShaheedBenazirabad',
   'Kashmor',
   ''],
  'Start_Date': ' July 2024',
  'End_Date': ' 1 September 2024',
  'Sentences': ' ['},
 'MDRCM039': {'Hazard': 'Flood',
  'Country': 'Cameroon',
  'Location': ['Yagoua', 'Blangoua', 'Mackary', 'Zina', 'Maga', ''],
  'Start_Date': ' July 202

In [156]:
#parse to a panda dataframe
import pandas as pd
result_df_list = []
for report_id, report in dict_processed.items():
    results_df = pd.DataFrame(report)
    results_df['report_id'] = report_id
    results_df.drop(['Sentences'], axis=1, inplace=True)
    result_df_list.append(results_df)
result_df_all = pd.concat(result_df_list)


In [160]:
result_df_all

,Hazard,Country,Location,Start_Date,End_Date,report_id
0,Flood,Algeria,Bchar,September 5 2024,NULL,MDRDZ011
1,Flood,Algeria,Elbayadh,September 5 2024,NULL,MDRDZ011
2,Flood,Algeria,BeniAbbes,September 5 2024,NULL,MDRDZ011
3,Flood,Algeria,Tamanrasset,September 5 2024,NULL,MDRDZ011
4,Flood,Algeria,Tiaret,September 5 2024,NULL,MDRDZ011
5,Flood,Algeria,Tindouf,September 5 2024,NULL,MDRDZ011
6,Flood,Algeria,Naama,September 5 2024,NULL,MDRDZ011
7,Flood,Algeria,,September 5 2024,NULL,MDRDZ011
0,Flood,Pakistan,Jacobabad,July 2024,1 September 2024,MDRPK026
1,Flood,Pakistan,NaushahroFeroz,July 2024,1 September 2024,MDRPK026
